# Capítulo 2 — Números, texto, datas e fusos horários

## Missão
Tratar os dados brutos que chegam do rover: arredondar e formatar números, limpar e montar textos, converter horários em UTC para o fuso de cada pessoa da equipe na Terra e calcular quanto tempo passou entre leituras.

**No fim deste caderno você terá:** o módulo `gabarito_rover/formatos.py`, com funções para formatar leituras e converter horários entre fusos.

**Leitura essencial (opcional, livro em inglês):**
- Calculating Numbers with Functions · PDF p. 105
- Formatting Numbers · PDF p. 111
- Manipulating Strings · PDF p. 121
- Uncovering Dates and Times · PDF p. 128
- Working with Time Zones · PDF p. 140

**Gabarito:** a mesma trilha com o código pronto. Tente primeiro em `praticar/`.

## Preparação
Configuração do caderno (já vem pronta). Rode antes de tudo.

In [ ]:
import os, sys, json, time, math, random
sys.path.insert(0, "../comum")                 # dados.py (dados da missão do rover)
sys.path.insert(0, "..")                       # raiz: onde fica o seu pacote gabarito_rover/
os.makedirs("../gabarito_rover", exist_ok=True)
open("../gabarito_rover/__init__.py", "a").close()        # gabarito_rover/ é o rover que você constrói ao longo do livro
os.makedirs("../saidas", exist_ok=True)        # arquivos gerados nos exercícios
from dados import *
%load_ext autoreload
%autoreload 2
print("Pronto. Python", sys.version.split()[0], "| seu pacote: gabarito_rover")

## Ideia em 30 segundos
- `round`, `abs`, `math` e as f-strings (`f"{x:.2f}"`) resolvem quase toda formatação de números.
- Textos têm dezenas de métodos (`strip`, `split`, `upper`, `replace`, `startswith`...).
- Para datas, o padrão é: guardar sempre em **UTC** e converter para o fuso local só na hora de mostrar (`zoneinfo`).

### 2.1 **Funções numéricas.** Com as temperaturas de `LEITURAS`, mostre a mínima, a máxima, a amplitude (`max - min`), a média arredondada em 1 casa e a raiz quadrada da amplitude (`math.sqrt`).

In [ ]:
temps = [l[1] for l in LEITURAS]
amplitude = max(temps) - min(temps)
print(min(temps), max(temps), round(amplitude, 1), round(sum(temps) / len(temps), 1), round(math.sqrt(amplitude), 2))

### 2.2 **Formatar números.** Mostre a distância 1250.5678 cm de 5 formas com f-strings: 2 casas, sem casas, com separador de milhar (`,`), em notação científica e como porcentagem de 5.000 cm (`:.1%`). Depois alinhe uma tabelinha de 3 leituras com largura fixa (`:>8`).

In [ ]:
d = 1250.5678
print(f"{d:.2f} | {d:.0f} | {d:,.2f} | {d:.3e} | {d / 5000:.1%}")
for hora, temp, poeira, bat, dist in LEITURAS[:3]:
    print(f"{hora[11:16]:>6} {temp:>8.1f} °C {bat:>5}% {dist:>6} cm")

### 2.3 **Números estranhos.** Mostre que `0.1 + 0.2 != 0.3`, resolva com `math.isclose` e com `decimal.Decimal("0.1")`. Use também `fractions.Fraction(1, 3) * 3`.

In [ ]:
from decimal import Decimal
from fractions import Fraction
print(0.1 + 0.2, 0.1 + 0.2 == 0.3, math.isclose(0.1 + 0.2, 0.3))
print(Decimal("0.1") + Decimal("0.2"), Fraction(1, 3) * 3)

### 2.4 **Textos.** A mensagem do rover chega assim: `"  status: OK ; bateria=87 ; modo=EXPLORAR  "`. Limpe os espaços, separe pelo `;`, transforme num dicionário (`chave: valor`, com `=` ou `:`) e mostre o modo em minúsculas. Por fim, junte as chaves numa só frase com `", ".join`.

In [ ]:
msg = "  status: OK ; bateria=87 ; modo=EXPLORAR  "
campos = {}
for parte in msg.strip().split(";"):
    chave, valor = parte.replace(":", "=").split("=", 1)
    campos[chave.strip()] = valor.strip()
print(campos, campos["modo"].lower())
print(", ".join(campos))

### 2.5 **Datas.** Converta a hora da primeira e da última leitura com `datetime.fromisoformat`. Calcule quanto tempo passou (`timedelta`), em horas, e mostre a data por extenso no formato `01/03/2026 06:00`.

In [ ]:
from datetime import datetime
inicio = datetime.fromisoformat(LEITURAS[0][0])
fim = datetime.fromisoformat(LEITURAS[-1][0])
duracao = fim - inicio
print(duracao, duracao.total_seconds() / 3600, "horas")
print(inicio.strftime("%d/%m/%Y %H:%M"))

### 2.6 **Fusos horários.** As leituras estão em UTC. Para cada pessoa de `EQUIPE`, mostre que horas eram na cidade dela quando chegou a leitura das 12:00 UTC (`ZoneInfo`). Quem estava dormindo (entre 23h e 6h)?

In [ ]:
from datetime import datetime, timezone
from zoneinfo import ZoneInfo
utc = datetime.fromisoformat("2026-03-01T12:00:00").replace(tzinfo=timezone.utc)
for p in EQUIPE:
    local = utc.astimezone(ZoneInfo(p["fuso"]))
    dormindo = local.hour >= 23 or local.hour < 6
    print(f"{p['nome']:6} {p['cidade']:8} {local:%d/%m %H:%M} {'(dormindo)' if dormindo else ''}")

### 2.7 **Módulo de formatos.** Escreva `formatos.py` com `formatar_leitura(leitura)`, que devolve uma linha alinhada com hora, temperatura, bateria e distância, e `para_fuso(hora_utc_iso, fuso)`, que devolve o `datetime` no fuso pedido.

_A célula grava `gabarito_rover/formatos.py`: a 1ª linha (`%%writefile`) já vem pronta, digite o código embaixo dela._

In [ ]:
%%writefile ../gabarito_rover/formatos.py
"""Formatação de leituras e conversão de fusos horários."""
from datetime import datetime, timezone
from zoneinfo import ZoneInfo


def formatar_leitura(leitura: tuple) -> str:
    hora, temp, poeira, bateria, dist = leitura
    return f"{hora[11:16]:>5} | {temp:>7.1f} °C | poeira {poeira:.2f} | {bateria:>3}% | {dist:>4} cm"


def para_fuso(hora_utc_iso: str, fuso: str) -> datetime:
    utc = datetime.fromisoformat(hora_utc_iso).replace(tzinfo=timezone.utc)
    return utc.astimezone(ZoneInfo(fuso))


### 2.8 **Use o módulo.** Mostre todas as leituras formatadas e a hora da última leitura no fuso de Tóquio.

In [ ]:
from gabarito_rover.formatos import formatar_leitura, para_fuso
for l in LEITURAS:
    print(formatar_leitura(l))
print(para_fuso(LEITURAS[-1][0], "Asia/Tokyo"))

## Fechamento

- O que eu construí hoje:
- O que me surpreendeu:
- O que quero tentar sozinho a seguir:
